In [1]:
import numpy as np
import tensorflow as tf
import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, Input
from tensorflow.keras.optimizers import Adam
from keras.losses import Huber

import random

In [2]:
def build_model(states: int, actions: int, hidden_layers=[24, 24]):
    """
    states - number of input nodes in a state 
    actions - number of actions
    """
    model = Sequential()
    
    # Flatten input layer (for handling state input shape)
    model.add(Input((1, states)))
    
    # Dynamically add hidden layers
    for neurons in hidden_layers:
        model.add(Dense(neurons, activation='relu'))
    
    # Output layer with softmax activation
    model.add(Dense(actions, activation='linear'))
    
    return model

In [ ]:
def reward_function(state, action):
    return ...
    

class Game:
    '''Representation of the game for training process'''

    def __init__(self):
        ...

    def do_action(self, action):
        new_state = ...
        reward = reward_function(...)
        return new_state, reward
    
    def get_curr_state(self):
        ...

    def action_space():
        return ...
    
    def is_finished():
        return False
    
    @property
    def num_actions(self):
        return len(self.action_space())


In [ ]:
# training
game = Game()
model = build_model(states=10, actions=4)
model_target = build_model(states=10, actions=4)
adam = Adam(learning_rate=0.00025, clipnorm=1.0)

# training params
epsilon = 1.0  # Epsilon greedy parameter
epsilon_min = 0.1  # Minimum epsilon greedy parameter
epsilon_max = 1.0  # Maximum epsilon greedy parameter
epsilon_step = (epsilon_max - epsilon_min) / 10_000 # Rate at which to reduce chance of random action being taken

steps_per_episode = 100
taget_update_period = 10
batch_size = 32
gamma = 0.7

loss_function = Huber()

while not game.is_finished():

    for episode in range(steps_per_episode):
        # initialise replay buffer
        buffer = []
        ACTION = 0
        STATE = 1
        REWARD = 2
        NEXT_STATE = 3
        FINISHED = 4

        for time_step in range(steps_per_episode):
            # epsilon greedy exploration
            p = random.random()
            epsilon = max(epsilon_min, epsilon - epsilon_step) # decreases random actions with every step by the epsilon_step
            if p > epsilon:
                curr_state = game.get_state()
                q_row = model.predict(curr_state) #FIXME: might not work due to the format of state, it's supposed to give the output layer
                action = keras.ops.argmax(q_row[0]).numpy()
            else: 
                action = game.action_space().sample() #TODO: get a random action (gonna depend on implementation of game class)

            # commit the action
            next_state, reward = game.do_action(action) # take the action and return the outcome

            # update replay buffer
            buffer.append((action, curr_state, reward, next_state))
        
        sample = np.array(random.choice(buffer, size=batch_size)) # randomly samples the buffer
        future_rewards = keras.ops.amax(model_target.predict(sample[:, NEXT_STATE]), axis=1) # for each next state, the max q value possible
        target = sample[:, REWARD] + (1 - sample[:, FINISHED]) * gamma * future_rewards # the funky formula

        masks = keras.ops.one_hot(sample[:, ACTION], game.num_actions)
        prediction_matrix = model.predict(sample[:, STATE])
        prediction = keras.ops.sum(keras.ops.multiply(prediction_matrix, masks), axis=1) # the prediction is an array of q values for the actions in the buffer sample

        # loss and backpropagation
        with tf.GradientTape() as tape:
            loss = loss_function(target, prediction)
            grads = tape.gradient(loss, model.trainable_variables)
            adam.apply_gradients(zip(grads, model.trainable_variables))


        # update the target model
        if episode % taget_update_period == 0:
            model_target.set_weights(model.get_weights())


